# Module 2 · ML Model Evaluation And Diagnostics

**Track:** AI Engineering Core  
**Objective:** Master the principles and applications of ML Model Evaluation And Diagnostics.

---



### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors optimize for a single metric. Seniors optimize for the metric that minimizes **Business Regret**. In medicine, we minimize False Negatives (Recall). In finance, we minimize False Positives (Precision). But to truly compare models, we use statistical significance tests like **McNemar's** to confirm that a 0.5% gain isn't just a lucky random seed.

**Mental Model:** 
- **Confusion Matrix:** The complete diagnostic CT-scan of your classifier. Every metric (Precision, Recall, F1, MCC) is just a different slice of this matrix.
- **ROC vs PR Curve:** ROC is the 'optimistic doctor' — it looks good even when things are bad. PR is the 'harsh auditor' — it exposes failure in imbalanced settings.
- **Cross-Validation:** Not just a scoring method — it's a **variance estimator**. The spread of fold scores tells you how stable your model is.
- **Calibration:** The difference between a model that ranks well and a model you can trust with dollar amounts.
- **McNemar's:** Looks at the 'Discordant Pairs' (where models disagree) to see if one model is actually logically superior.

**Common Junior Pitfall:** Assuming that if Model B has 0.1% higher AUC than Model A, it's better. At scale, this difference is often noise. Never swap a model in production without a statistical $p < 0.05$ result.

---


## 📑 Table of Contents

  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1. The Confusion Matrix — Your Diagnostic Foundation](#1-the-confusion-matrix-your-diagnostic-foundation)
  * [1.1 Multi-Class Confusion Matrix](#11-multi-class-confusion-matrix)
* [2. ROC & Precision-Recall Curves — Beyond Single Scores](#2-roc-precision-recall-curves-beyond-single-scores)
  * [2.1 The Imbalance Trap](#21-the-imbalance-trap)
* [3. Beyond F1: Cohen's Kappa & MCC](#3-beyond-f1-cohens-kappa-mcc)
* [4. Cross-Validation — The Variance Estimator](#4-cross-validation-the-variance-estimator)
  * [4.1 Stratified, Time Series, and Grouped Splits](#41-stratified-time-series-and-grouped-splits)
  * [4.2 Nested Cross-Validation](#42-nested-cross-validation)
* [5. Learning Curves — Bias vs Variance Diagnosis](#5-learning-curves-bias-vs-variance-diagnosis)
* [6. Calibration — When Confidence Must Mean Something](#6-calibration-when-confidence-must-mean-something)
  * [6.1 Platt Scaling vs Isotonic Regression](#61-platt-scaling-vs-isotonic-regression)
* [7. Threshold Optimization — Business-Aware Decisions](#7-threshold-optimization-business-aware-decisions)
* [8. Regression Diagnostics — Beyond MSE](#8-regression-diagnostics-beyond-mse)
* [9. Statistical Model Comparison — McNemar's Test](#9-statistical-model-comparison-mcnemars-test)
* [✅ Knowledge Check](#knowledge-check)
* [🔨 Practical Practice](#practical-practice)


## 1. The Confusion Matrix — Your Diagnostic Foundation

Every classification metric is a **projection** of the confusion matrix. Understanding it is non-negotiable.

| | Predicted Positive | Predicted Negative |
|:---|:---|:---|
| **Actual Positive** | True Positive (TP) | False Negative (FN) — Type II Error |
| **Actual Negative** | False Positive (FP) — Type I Error | True Negative (TN) |

**Key derived metrics:**
- **Precision** = $\frac{TP}{TP + FP}$ — "Of all I flagged, how many were real?"
- **Recall (Sensitivity)** = $\frac{TP}{TP + FN}$ — "Of all the real ones, how many did I catch?"
- **Specificity** = $\frac{TN}{TN + FP}$ — "Of all negatives, how many did I correctly ignore?"
- **F1** = $\frac{2 \cdot P \cdot R}{P + R}$ — Harmonic mean (punishes imbalance between P and R)

📈 **Production Signal:** In fraud detection, a 0.1% improvement in Recall can save millions. In spam filtering, a 0.1% drop in Precision means legitimate emails get lost. The business context dictates which metric is king.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, 
                             classification_report, f1_score,
                             roc_auc_score, average_precision_score)

# Load real dataset
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Train a baseline model
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

# --- Confusion Matrix Heatmap ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=['Malignant', 'Benign'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title('Confusion Matrix (Counts)')

# Normalized (row-wise = recall per class)
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=['Malignant', 'Benign'],
    normalize='true', cmap='Oranges', ax=axes[1]
)
axes[1].set_title('Confusion Matrix (Normalized by True Label)')
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=['Malignant', 'Benign']))

"""What just happened?
We trained a Random Forest on the Breast Cancer dataset and visualized the confusion
matrix in two ways: raw counts (left) and normalized by true label (right). The 
normalized view immediately reveals per-class recall — critical for medical diagnosis
where missing a malignant tumor (FN) is far worse than a false alarm (FP).
"""

### 1.1 Multi-Class Confusion Matrix

Binary classification is the easy case. In production, you often have 10+ classes. The confusion matrix becomes a **heat-map**, and per-class metrics become essential.


In [ ]:
from sklearn.datasets import load_digits
from sklearn.metrics import ConfusionMatrixDisplay

digits = load_digits()
X_d_train, X_d_test, y_d_train, y_d_test = train_test_split(
    digits.data, digits.target, test_size=0.3, random_state=42, stratify=digits.target
)

rf_mc = RandomForestClassifier(n_estimators=100, random_state=42)
rf_mc.fit(X_d_train, y_d_train)
y_mc_pred = rf_mc.predict(X_d_test)

fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay.from_predictions(
    y_d_test, y_mc_pred, cmap='YlOrRd', ax=ax,
    colorbar=False
)
ax.set_title('10-Class Confusion Matrix (Handwritten Digits)', fontsize=14)
plt.tight_layout()
plt.show()

# Per-class F1
print(classification_report(y_d_test, y_mc_pred))

"""What just happened?
We built a 10-class confusion matrix for digit recognition. Off-diagonal cells reveal
systematic confusion patterns: 3 vs 8, 4 vs 9, etc. In production, these patterns
guide targeted data collection — if the model confuses 4s and 9s, we need more
training examples that disambiguate the two.
"""

## 2. ROC & Precision-Recall Curves — Beyond Single Scores

A single AUC number hides critical information. The **curve** shows you how your model trades off errors at every possible threshold.

- **ROC Curve (FPR vs TPR):** Shows how Recall changes as you allow more False Positives.
- **PR Curve (Recall vs Precision):** Shows how Precision degrades as you try to catch more positives.

🎓 **Socratic Deep Check:**
> *"ROC-AUC can be 0.95 even when your model is practically useless. How? When 99.9% of your data is negative, even random scoring gives a low FPR because the FP count is tiny relative to the massive TN count."*


In [ ]:
from sklearn.metrics import RocCurveDisplay, PrecisionRecallDisplay

# Compare 3 models on the same data
models = {
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=5000, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = ['#e74c3c', '#3498db', '#2ecc71']

for (name, model), color in zip(models.items(), colors):
    model.fit(X_train, y_train)
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=axes[0], 
                                   name=name, color=color)
    PrecisionRecallDisplay.from_estimator(model, X_test, y_test, ax=axes[1],
                                          name=name, color=color)

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random Classifier')
axes[0].set_title('ROC Curves — Multi-Model Comparison', fontsize=13)
axes[1].set_title('Precision-Recall Curves — Multi-Model Comparison', fontsize=13)

for ax in axes:
    ax.legend(loc='lower left')
plt.tight_layout()
plt.show()

"""What just happened?
We plotted ROC and PR curves for three different classifiers on the same test set.
The ROC curves look similar (all clustered near top-left). The PR curves often reveal
more nuance — a model with perfect ROC-AUC can still have poor precision at high
recall thresholds. Always plot BOTH.
"""

### 2.1 The Imbalance Trap

ROC-AUC is dangerously optimistic on imbalanced data because it is insensitive to the sheer volume of False Positives. Precision-Recall AUC (PR-AUC) captures the true cost of failure.


In [ ]:
# Simulate extreme imbalance (99% negative)
np.random.seed(42)
y_true_imb = np.array([0]*990 + [1]*10)
y_scores_imb = np.random.rand(1000)
y_scores_imb[y_true_imb == 1] += 0.5  # Give positives a slight bump
y_scores_imb = np.clip(y_scores_imb, 0, 1)

roc = roc_auc_score(y_true_imb, y_scores_imb)
pr = average_precision_score(y_true_imb, y_scores_imb)

print(f"ROC-AUC: {roc:.3f} (Looks Great!)")
print(f"PR-AUC:  {pr:.3f} (The Harsh Reality)")
print(f"\nThe gap of {roc - pr:.3f} is the 'Imbalance Illusion'.")
print("At 99% negative, random baseline PR-AUC ≈ 0.01, not 0.50.")

"""What just happened?
ROC-AUC says the model looks good (>0.7), but PR-AUC reveals that almost every
positive prediction is a false alarm. In fraud detection with 0.1% fraud rate,
ROC-AUC is nearly meaningless — PR-AUC is the metric that matters.
"""

## 3. Beyond F1: Cohen's Kappa & MCC

**Cohen's Kappa ($\kappa$):** Adjusts accuracy for chance agreement:
$$\kappa = \frac{p_o - p_e}{1 - p_e}$$
Where $p_o$ is observed accuracy and $p_e$ is expected accuracy by random chance.

**Matthews Correlation Coefficient (MCC):**
$$MCC = \frac{TP \cdot TN - FP \cdot FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$
MCC is the **gold standard** for binary classification on imbalanced data. It ranges from -1 (complete disagreement) to +1 (perfect) and considers all four quadrants equally.


In [ ]:
from sklearn.metrics import matthews_corrcoef, cohen_kappa_score

# Demonstrate the trap: an "always majority" classifier
y_true_skew = np.array([0]*950 + [1]*50)
y_pred_majority = np.zeros(1000)  # Always predicts negative
y_pred_decent = np.array([0]*940 + [1]*60)
y_pred_decent[:950] = np.where(np.random.rand(950) > 0.02, 0, 1)
y_pred_decent[950:] = np.where(np.random.rand(50) > 0.3, 1, 0)

print("=== 'Always Negative' Classifier ===")
print(f"  Accuracy:  {(y_true_skew == y_pred_majority).mean():.3f} (Looks amazing!)")
print(f"  F1:        {f1_score(y_true_skew, y_pred_majority, zero_division=0):.3f}")
print(f"  MCC:       {matthews_corrcoef(y_true_skew, y_pred_majority):.3f}")
print(f"  Kappa:     {cohen_kappa_score(y_true_skew, y_pred_majority):.3f}")

print("\n=== Decent Classifier ===")
print(f"  Accuracy:  {(y_true_skew == y_pred_decent).mean():.3f}")
print(f"  F1:        {f1_score(y_true_skew, y_pred_decent):.3f}")
print(f"  MCC:       {matthews_corrcoef(y_true_skew, y_pred_decent):.3f}")
print(f"  Kappa:     {cohen_kappa_score(y_true_skew, y_pred_decent):.3f}")

"""What just happened?
The 'always negative' model gets 95% accuracy but MCC=0 and Kappa=0, correctly 
identifying it as useless. MCC and Kappa see through the majority-class gaming that
simple accuracy misses. In production, always report MCC alongside F1.
"""

## 4. Cross-Validation — The Variance Estimator

A single train/test split is a **single sample** from the distribution of possible model performances. Cross-validation gives us the **mean AND variance** of that distribution.

**Why this matters:**
- Model A: 5-fold scores = [0.91, 0.92, 0.90, 0.91, 0.92] → Stable ✅
- Model B: 5-fold scores = [0.95, 0.85, 0.99, 0.80, 0.92] → Unstable ❌

Model B has a higher mean but is unreliable. A senior would pick Model A.


In [ ]:
from sklearn.model_selection import (cross_val_score, KFold, StratifiedKFold,
                                     TimeSeriesSplit, RepeatedStratifiedKFold)

# Compare CV strategies on Breast Cancer
model = RandomForestClassifier(n_estimators=100, random_state=42)

cv_strategies = {
    'KFold (k=5)': KFold(n_splits=5, shuffle=True, random_state=42),
    'StratifiedKFold (k=5)': StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    'Repeated Stratified (5×3)': RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42),
}

fig, ax = plt.subplots(figsize=(10, 5))
all_scores = []
labels = []

for name, cv in cv_strategies.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring='f1')
    all_scores.append(scores)
    labels.append(name)
    print(f"{name:30s} | Mean F1: {scores.mean():.4f} ± {scores.std():.4f}")

ax.boxplot(all_scores, labels=labels)
ax.set_ylabel('F1 Score')
ax.set_title('Cross-Validation Strategy Comparison', fontsize=13)
plt.tight_layout()
plt.show()

"""What just happened?
We compared three CV strategies. StratifiedKFold preserves class distribution in
every fold — essential for imbalanced data. RepeatedStratifiedKFold runs multiple
repetitions with different random splits, giving the tightest confidence interval.
The box plot visually shows which strategy produces the most stable estimates.
"""

### 4.1 Stratified, Time Series, and Grouped Splits

| Strategy | Use Case | Data Leakage Risk |
|:---|:---|:---|
| `KFold` | General purpose, balanced data | Low |
| `StratifiedKFold` | Imbalanced classification | Low |
| `TimeSeriesSplit` | Temporal data (stocks, logs) | **Critical** — never shuffle! |
| `GroupKFold` | Multiple samples per entity (patients, users) | **Critical** — groups must not leak |

> **⚠️ Production Warning:** Using standard KFold on time series data is the #1 cause of overly optimistic model evaluations in production ML. Future data leaks into training folds, giving you a model that 'remembers' the future.


### 4.2 Nested Cross-Validation

When you do hyperparameter tuning inside CV, you are **optimizing on the test fold**, biasing the score upward. **Nested CV** fixes this:

- **Inner loop:** Hyperparameter tuning (e.g., GridSearchCV)
- **Outer loop:** Unbiased performance estimation

This is the **gold standard** for model selection in academic papers and rigorous production ML.


In [ ]:
from sklearn.model_selection import cross_val_score, GridSearchCV

# NON-nested: optimistic estimate
param_grid = {'max_depth': [3, 5, 10, None], 'n_estimators': [50, 100, 200]}
gs = GridSearchCV(RandomForestClassifier(random_state=42), param_grid, cv=3, scoring='f1')
non_nested = cross_val_score(gs, X, y, cv=5, scoring='f1')  # Outer CV

# Simple (non-tuned) baseline for comparison
simple = cross_val_score(RandomForestClassifier(random_state=42), X, y, cv=5, scoring='f1')

print(f"Non-nested (tuned) CV F1:  {non_nested.mean():.4f} ± {non_nested.std():.4f}")
print(f"Simple (default) CV F1:    {simple.mean():.4f} ± {simple.std():.4f}")
print(f"\nThe nested score gives an UNBIASED estimate of generalization.")
print(f"If the non-nested score is much higher than nested, you are overfitting hyperparameters.")

"""What just happened?
We used nested CV (GridSearchCV inside cross_val_score). The outer loop gives an
unbiased estimate, while the inner loop tunes hyperparameters. If the gap between
nested and non-nested scores is large, your hyperparameter search is overfitting.
"""

## 5. Learning Curves — Bias vs Variance Diagnosis

Learning curves answer the fundamental question: **"Will more data help?"**

- **High Bias (Underfitting):** Both train and validation scores are low and converging → More data won't help. You need a more complex model.
- **High Variance (Overfitting):** Train score is high, validation is low, with a gap → More data WILL help. Or use regularization.


In [ ]:
from sklearn.model_selection import learning_curve

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

models_lc = [
    ('High Variance: Deep Tree (max_depth=None)', 
     RandomForestClassifier(n_estimators=10, max_depth=None, random_state=42)),
    ('Low Variance: Shallow Tree (max_depth=2)', 
     RandomForestClassifier(n_estimators=10, max_depth=2, random_state=42))
]

for ax, (title, model) in zip(axes, models_lc):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X, y, cv=5, train_sizes=np.linspace(0.1, 1.0, 8),
        scoring='f1', random_state=42
    )
    ax.fill_between(train_sizes, train_scores.mean(axis=1) - train_scores.std(axis=1),
                    train_scores.mean(axis=1) + train_scores.std(axis=1), alpha=0.1, color='#e74c3c')
    ax.fill_between(train_sizes, val_scores.mean(axis=1) - val_scores.std(axis=1),
                    val_scores.mean(axis=1) + val_scores.std(axis=1), alpha=0.1, color='#3498db')
    ax.plot(train_sizes, train_scores.mean(axis=1), 'o-', color='#e74c3c', label='Training')
    ax.plot(train_sizes, val_scores.mean(axis=1), 'o-', color='#3498db', label='Validation')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Training Set Size')
    ax.set_ylabel('F1 Score')
    ax.legend(loc='lower right')
    ax.set_ylim([0.7, 1.05])

plt.suptitle('Learning Curves: Diagnosing Bias vs Variance', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

"""What just happened?
Left: Deep trees overfit — training score is perfect but validation lags behind (the
gap = variance). More data would help close this gap. Right: Shallow trees underfit —
both scores converge at a low ceiling (bias). More data cannot help; you need more
model capacity.
"""

## 6. Calibration — When Confidence Must Mean Something

A model is **Calibrated** if its predicted probability equals the actual frequency: if you predict 80% confidence for 100 patients, ~80 should actually be positive.

**Expected Calibration Error (ECE)** bins predictions by confidence and measures the weighted gap between confidence and accuracy:
$$ECE = \sum_{b=1}^{B} \frac{|B_b|}{n} |\text{acc}(B_b) - \text{conf}(B_b)|$$

📈 **Production Signal:** If your business logic triggers actions at specific probability thresholds (e.g., "send coupon if churn_prob > 0.9"), an uncalibrated model may never reach 0.9 despite having high ranking capability.


In [ ]:
from sklearn.calibration import CalibrationDisplay, CalibratedClassifierCV

# Train models known for different calibration profiles
models_cal = {
    'Random Forest (often uncalibrated)': RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression (well-calibrated)': LogisticRegression(max_iter=5000, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors = ['#e74c3c', '#3498db', '#2ecc71']

for (name, model), color in zip(models_cal.items(), colors):
    model.fit(X_train, y_train)
    CalibrationDisplay.from_estimator(model, X_test, y_test, n_bins=10,
                                      ax=axes[0], name=name, color=color)

axes[0].set_title('Reliability Diagram (Before Calibration)', fontsize=13)

# --- 6.1 Fix calibration with Platt and Isotonic ---
rf_uncal = RandomForestClassifier(n_estimators=100, random_state=42)
rf_platt = CalibratedClassifierCV(rf_uncal, method='sigmoid', cv=5)
rf_isotonic = CalibratedClassifierCV(rf_uncal, method='isotonic', cv=5)

for model, name, color in [
    (RandomForestClassifier(n_estimators=100, random_state=42), 'RF (Original)', '#e74c3c'),
    (CalibratedClassifierCV(RandomForestClassifier(n_estimators=100, random_state=42), 
                            method='sigmoid', cv=5), 'RF + Platt Scaling', '#9b59b6'),
    (CalibratedClassifierCV(RandomForestClassifier(n_estimators=100, random_state=42), 
                            method='isotonic', cv=5), 'RF + Isotonic', '#f39c12'),
]:
    model.fit(X_train, y_train)
    CalibrationDisplay.from_estimator(model, X_test, y_test, n_bins=10,
                                      ax=axes[1], name=name, color=color)

axes[1].set_title('After Calibration: Platt vs Isotonic', fontsize=13)
plt.tight_layout()
plt.show()

"""What just happened?
Left: Random Forest pushes probabilities toward 0.5 (the 'averaging' effect of
many trees), making it poorly calibrated. Logistic Regression is naturally calibrated.
Right: Platt Scaling (sigmoid fit) and Isotonic Regression fix the RF calibration.
Isotonic is more flexible but can overfit with small datasets.
"""

## 7. Threshold Optimization — Business-Aware Decisions

The default threshold of 0.5 is **arbitrary**. In production, the optimal threshold depends on the **asymmetric costs** of False Positives vs False Negatives.

**Example:** In cancer screening, missing a malignant tumor (FN) costs a life. A false alarm (FP) costs a biopsy. The optimal threshold should heavily favor recall.


In [ ]:
from sklearn.metrics import precision_recall_curve, f1_score

# Use Gradient Boosting for well-separated probabilities
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)
y_prob = gb.predict_proba(X_test)[:, 1]

# Sweep thresholds
thresholds = np.linspace(0.01, 0.99, 200)
precisions, recalls, f1s = [], [], []

for t in thresholds:
    y_t = (y_prob >= t).astype(int)
    precisions.append((y_t[y_test == 1] == 1).sum() / max(y_t.sum(), 1))
    recalls.append((y_t[y_test == 1] == 1).sum() / max((y_test == 1).sum(), 1))
    f1s.append(f1_score(y_test, y_t, zero_division=0))

best_t = thresholds[np.argmax(f1s)]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(thresholds, precisions, label='Precision', color='#e74c3c', linewidth=2)
ax.plot(thresholds, recalls, label='Recall', color='#3498db', linewidth=2)
ax.plot(thresholds, f1s, label='F1', color='#2ecc71', linewidth=2.5)
ax.axvline(0.5, color='gray', linestyle='--', alpha=0.5, label='Default (0.5)')
ax.axvline(best_t, color='#f39c12', linestyle='--', linewidth=2, label=f'Optimal F1 ({best_t:.2f})')
ax.set_xlabel('Classification Threshold', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Threshold Optimization: Precision/Recall/F1 vs Threshold', fontsize=14)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print(f"Optimal F1 threshold: {best_t:.3f} (vs default 0.5)")
print(f"F1 at default:  {f1s[np.argmin(np.abs(thresholds - 0.5))]:.4f}")
print(f"F1 at optimal:  {max(f1s):.4f}")

"""What just happened?
We swept all possible thresholds and plotted the resulting precision, recall, and F1.
The optimal threshold is rarely 0.5. In cancer detection, you might set it at 0.2
to maximize recall (catch all tumors), accepting more false alarms (biopsies).
"""

## 8. Regression Diagnostics — Beyond MSE

For regression tasks, a single MSE/MAE number hides systematic problems. **Residual analysis** reveals:
- **Heteroscedasticity:** Variance changes with the predicted value (violates OLS assumptions).
- **Non-linearity:** Patterns in residuals indicate the model is missing a relationship.
- **Outlier influence:** A handful of extreme points may dominate the loss.


In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy import stats

housing = fetch_california_housing()
Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    housing.data, housing.target, test_size=0.3, random_state=42
)

reg = GradientBoostingRegressor(n_estimators=200, max_depth=4, random_state=42)
reg.fit(Xh_train, yh_train)
yh_pred = reg.predict(Xh_test)
residuals = yh_test - yh_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Residuals vs Predicted
axes[0].scatter(yh_pred, residuals, alpha=0.2, s=10, color='#3498db')
axes[0].axhline(0, color='red', linestyle='--')
axes[0].set_xlabel('Predicted Value')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted (Check Heteroscedasticity)')

# 2. Q-Q Plot
stats.probplot(residuals, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot (Check Normality of Residuals)')

# 3. Residual Distribution
axes[2].hist(residuals, bins=50, color='#2ecc71', alpha=0.7, edgecolor='white')
axes[2].axvline(0, color='red', linestyle='--')
axes[2].set_xlabel('Residual')
axes[2].set_title('Residual Distribution (Should be ~Normal, centered at 0)')

plt.suptitle('Regression Diagnostics: California Housing', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"MSE:  {mean_squared_error(yh_test, yh_pred):.4f}")
print(f"MAE:  {mean_absolute_error(yh_test, yh_pred):.4f}")
print(f"R²:   {r2_score(yh_test, yh_pred):.4f}")

"""What just happened?
Three diagnostic plots: (1) Residuals vs Predicted reveals heteroscedasticity — the
fan shape means variance increases with price, common in housing data. (2) Q-Q plot
shows heavy tails — extreme residuals are more common than a Normal distribution
predicts. (3) The histogram confirms the non-normality. These diagnostics guide
decisions about loss functions (MAE for heavy tails) and transforms (log-price).
"""

## 9. Statistical Model Comparison — McNemar's Test

When Model A has 85.3% accuracy and Model B has 85.7%, is Model B truly better?
**McNemar's test** looks only at the points where Models A and B **disagree**:

| | B Correct | B Wrong |
|:---|:---|:---|
| **A Correct** | $n_{00}$ (both right) | $n_{01}$ (only A right) |
| **A Wrong** | $n_{10}$ (only B right) | $n_{11}$ (both wrong) |

The test statistic: $\chi^2 = \frac{(|n_{01} - n_{10}| - 1)^2}{n_{01} + n_{10}}$

If $p > 0.05$: the models are **not significantly different**. Do not swap your champion.


In [ ]:
from scipy.stats import binom_test

# Train two models
rf_a = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_train, y_train)
rf_b = RandomForestClassifier(n_estimators=100, random_state=43).fit(X_train, y_train)

pred_a = rf_a.predict(X_test)
pred_b = rf_b.predict(X_test)

# Build contingency table
a_correct = (pred_a == y_test)
b_correct = (pred_b == y_test)

n01 = np.sum(a_correct & ~b_correct)  # Only A correct
n10 = np.sum(~a_correct & b_correct)  # Only B correct

print(f"Model A accuracy: {a_correct.mean():.4f}")
print(f"Model B accuracy: {b_correct.mean():.4f}")
print(f"\nDisagreement pairs: A-only={n01}, B-only={n10}")

# McNemar's exact test (binomial)
if n01 + n10 > 0:
    p_value = binom_test(n01, n01 + n10, 0.5)
    print(f"McNemar's p-value: {p_value:.4f}")
    if p_value > 0.05:
        print("→ NOT significantly different. Keep the champion model.")
    else:
        print("→ Significantly different. The challenger wins.")
else:
    print("Models agree on ALL predictions — no disagreement to test.")

"""What just happened?
We compared two Random Forest models differing only in random seed. Despite small
accuracy differences, McNemar's test correctly identifies that the difference is NOT
statistically significant — it's just random seed noise. Never deploy a model swap
based on noise.
"""

---
## ✅ Knowledge Check

### Q1: MCC vs. F1 on imbalance
<details><summary>👉 Answer</summary>
F1 score does not use True Negatives (TN). If a model predicts 'Negative' for everything, F1 is 0 but Accuracy is 95%+. MCC uses all four quadrants of the confusion matrix — it will only be high if the model predicts BOTH classes correctly. MCC is the only metric invariant to class swap.
</details>

### Q2: When does more data NOT help?
<details><summary>👉 Answer</summary>
When the learning curve shows HIGH BIAS (underfitting). Both training and validation scores converge at a low level. The model lacks capacity to fit the data, so adding more data just reinforces the same ceiling. Solution: increase model complexity, add features, or reduce regularization.
</details>

### Q3: Why calibrate a model?
<details><summary>👉 Answer</summary>
If your business logic triggers a $50 coupon when churn probability > 0.9, an uncalibrated model might never hit 0.9 despite having high ranking capability. Calibration ensures the model's 'confidence' has a physical meaning. Random Forests are notoriously poorly calibrated because they average tree votes, pushing probabilities toward 0.5.
</details>

### Q4: Why not just use accuracy?
<details><summary>👉 Answer</summary>
In a dataset with 99% negatives, an 'always negative' classifier gets 99% accuracy. Accuracy is meaningless on imbalanced data. Use PR-AUC, MCC, or F1 with the minority class as positive.
</details>


---
## 🔨 Practical Practice

### Tier 1: Beginner
1. Build a confusion matrix for a 3-class problem and manually calculate Precision, Recall, and F1 for each class.
2. Modify the threshold optimization code to find the threshold that maximizes **Recall** while keeping Precision above 0.8.

### Tier 2: Intermediate
1. **Time Series CV Trap:** Generate a synthetic time series with a trend. Show that `KFold` gives an optimistically high score while `TimeSeriesSplit` gives a realistic (lower) score.
2. **Calibration Improvement:** Train an XGBoost model on the Breast Cancer dataset. Plot its reliability diagram, then apply Isotonic calibration and show the ECE improvement numerically.
3. **Cost-Sensitive Threshold:** Given that a False Negative costs $1000 and a False Positive costs $10, derive the optimal threshold mathematically and verify it matches your threshold sweep.

### Tier 3: Advanced
1. **Nested CV Bias Quantification:** Run both nested and non-nested CV on 5 different datasets. Measure the 'optimism bias' (non-nested score minus nested score) for each. Is the bias consistent?
2. **Bayesian Model Comparison:** Instead of McNemar's frequentist test, implement a Bayesian correlated t-test for paired cross-validation (Benavoli et al., 2017). Compare the credible intervals to McNemar's p-value.
3. **Multi-Metric Pareto Front:** For a given model, sweep all thresholds and plot the Precision-Recall-FPR as a 3D Pareto front. Identify the set of 'non-dominated' thresholds.
